<a href="https://colab.research.google.com/github/busycaesar/GPT/blob/Master/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Download the dataset to train on.
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

--2026-08-23 09:42:44--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.04s   

2026-08-23 09:42:44 (28.2 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [2]:
print("Total characters:", len(text))

Total characters: 1115394


In [36]:
# All the unique characters that occur in the text
unique_characters = sorted(list(set(text)))
vocab_size = len(unique_characters)

In [38]:
print("".join(unique_characters))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [5]:
# Mapping for each unique character.
stoi = { ch:i for i,ch in enumerate(unique_characters) }
itos = { i:ch for i,ch in enumerate(unique_characters) }

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

In [6]:
print(encode("hii there"))
print(decode(encode("hii there")))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [7]:
import torch

encoded_text = encode(text)

dataset = torch.tensor(encoded_text, dtype=torch.long)

In [8]:
print(dataset.shape, dataset.dtype)
print(dataset[:1000])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

In [9]:
# Split the data into train and validation sets

train_dataset_proportion = 0.9

number_of_dataset = int(train_dataset_proportion*len(dataset))

train_dataset = dataset[:number_of_dataset]
validation_dataset = dataset[number_of_dataset:]

In [10]:
block_size = 8

train_dataset[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [19]:
x = train_dataset[:block_size]
y = train_dataset[1:block_size+1]

for tensor in range(block_size):
    context = x[:tensor+1]
    target = y[tensor]
    print(f"When input is {context} the expected target is {target}.")

When input is tensor([18]) the expected output is 47.
When input is tensor([18, 47]) the expected output is 56.
When input is tensor([18, 47, 56]) the expected output is 57.
When input is tensor([18, 47, 56, 57]) the expected output is 58.
When input is tensor([18, 47, 56, 57, 58]) the expected output is 1.
When input is tensor([18, 47, 56, 57, 58,  1]) the expected output is 15.
When input is tensor([18, 47, 56, 57, 58,  1, 15]) the expected output is 47.
When input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the expected output is 58.


In [66]:
# To maintain reproducability of random indices.
torch.manual_seed(1337)

# Parallel process on GPU
batch_size = 4
# Maximum context length for predicting next token
block_size = 8

def get_batch(split):
    dataset = train_dataset if split == 'train' else validation_dataset
    # Returns "batch_size" (4) random starting indices from the dataset.
    # The upper bound is "len(dataset) - block_size" (1003854 - 8) so that there are enough tokens for the block size even if the largest possible index is picked.
    ix = torch.randint(len(dataset) - block_size, (batch_size,))

    # Get "block_size" tokens starting at each chosen index, for all indices.
    context = torch.stack([dataset[i:i+block_size] for i in ix])

    # Get "block_size" tokens starting one position after each chosen index, for all indices.
    target = torch.stack([dataset[i+1:i+block_size+1] for i in ix])
    return context, target

In [65]:
context_ids, target_ids = get_batch('train')
print('inputs:')
print(context_ids)
print()
print('targets:')
print(target_ids)

print('----')

inputs:
tensor([[ 6,  0, 21, 44,  1, 61, 43,  1],
        [58, 52, 43, 57, 57,  2,  1, 57],
        [ 1, 59, 52, 39, 41, 46, 47, 52],
        [43, 42, 50, 39, 56,  6,  1, 50]])

targets:
tensor([[ 0, 21, 44,  1, 61, 43,  1, 61],
        [52, 43, 57, 57,  2,  1, 57, 43],
        [59, 52, 39, 41, 46, 47, 52, 45],
        [42, 50, 39, 56,  6,  1, 50, 43]])
----


In [67]:
for batch in range(batch_size):
    for tensor in range(block_size):
        context = context_ids[batch, :tensor+1]
        target = target_ids[batch, tensor]
        print(f"when input is {context.tolist()} the target: {target}")

when input is [6] the target: 0
when input is [6, 0] the target: 21
when input is [6, 0, 21] the target: 44
when input is [6, 0, 21, 44] the target: 1
when input is [6, 0, 21, 44, 1] the target: 61
when input is [6, 0, 21, 44, 1, 61] the target: 43
when input is [6, 0, 21, 44, 1, 61, 43] the target: 1
when input is [6, 0, 21, 44, 1, 61, 43, 1] the target: 61
when input is [58] the target: 52
when input is [58, 52] the target: 43
when input is [58, 52, 43] the target: 57
when input is [58, 52, 43, 57] the target: 57
when input is [58, 52, 43, 57, 57] the target: 2
when input is [58, 52, 43, 57, 57, 2] the target: 1
when input is [58, 52, 43, 57, 57, 2, 1] the target: 57
when input is [58, 52, 43, 57, 57, 2, 1, 57] the target: 43
when input is [1] the target: 59
when input is [1, 59] the target: 52
when input is [1, 59, 52] the target: 39
when input is [1, 59, 52, 39] the target: 41
when input is [1, 59, 52, 39, 41] the target: 46
when input is [1, 59, 52, 39, 41, 46] the target: 47
when

In [71]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# To maintain reproducability of random weights.
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # Lookup table mapping each token id to a row of vocab_size numbers. Each cell is assigned random values before training.
        # Normally, each token id is mapped to the row that contains the embedding (carrying semantic meaning) of the token.
        # The embeddings are then converted into logits to predict the next token.
        # Here, we skip all of that. The row length already equals vocab_size, so the row can be used directly as the logits.
        # So this model learns the logits directly in the table, instead of learning the many layers of weights that a real model uses to produce them.
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, context_ids, targets=None):
        # Gets the context batch matrix and returns logits array (vocab_size (64) size) for each index in the matrix.
        logits = self.token_embedding_table(context_ids) # Logit's Shape = Batch Size: B, Block Size: T, Vocab Size: C)

        if targets is None:
            return logits, None

        # Calculate the loss, based on the targets.

        B, T, C = logits.shape

        # Flattens the first two dimensions into one, so the shape goes from (Batch Size, Block Size, Vocab Size) to (Batch Size * Block Size, Vocab Size).
        #               Batch Item 1      Batch Item 2      Batch Item 3      Batch Item 4
        #               ________________  ________________  ________________  _________________
        # For example, [[[1, 2], [2, 3]], [[3, 4], [4, 5]], [[5, 6], [6, 7]], [[7, 8], [8, 9]]] (three nested arrays) becomes
        #               ______________  ______________  ______________  ______________
        #              [[1, 2], [2, 3], [3, 4], [4, 5], [5, 6], [6, 7], [7, 8], [8, 9]] (two nested arrays).
        # Essentially, the rows of all the batch items are joined into one flat list.
        # This is needed because the loss function expects the logits in the shape (Predictions, Vocab Size).
        logits = logits.view(B*T, C)

        # Converting the target matrix also into the same dimension as logits.
        targets = targets.view(B*T)

        loss = F.cross_entropy(logits, targets)

        # The cross enthropy gets the first item from the targets array, which is the expected index of the next token id.
        next_token_id_index = targets[0]

        # Then it gets the first logit array, in which it checks for the value on the expected index of the next token id.
        value_at_expected_index = logits[0][next_token_id_index]

        print()
        print("Next expected token id's index   :",next_token_id_index)
        print("Logit value at the expected index:",value_at_expected_index)
        print()

        # The cross enthropy then applies softmax to all the items in the array at 0th index of the logits array.
        # Then gets the softmaxed value (probability) at the expected index, and applies negative log to the probability.
        # Log of any probability (0-1) produces a negative number.
        # For higher probability (e.g., 0.9), log returns smaller negative (e.g., -0.105) and vise-versa. Negation converts to positive: -log(prob) gives positive loss.
        # For higher probability, we get lower loss value and vise-versa. So the correct predictions are rewarded.
        # It calculates the loss for each token prediction and returns the average loss.

        return logits, loss

    def generate(self, context_ids, max_new_tokens):
        # context_ids' shape = (B, T) array of indices.
        for _ in range(max_new_tokens):
            # Get the predictions for all the context in all the batch.
            logits, _ = self(context_ids)
            # Get the logits of the last predicated token.
            last_logit = logits[:, -1, :]
            # Apply softmax to all the numbers in the last logit to get the probabilities.
            probabilities = F.softmax(last_logit, dim=-1)
            # sample from the distribution
            index_of_next_token = torch.multinomial(probabilities, num_samples=1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, index_of_next_token), dim=1)
        return idx

In [72]:
m = BigramLanguageModel(vocab_size)
print("Context Id Batch")
print(context_ids)
print()
print("Target Id Batch")
print(target_ids)
print()
logits, loss = m(context_ids, target_ids)
print(logits.shape)
print(loss)
print()

Context Id Batch
tensor([[ 6,  0, 21, 44,  1, 61, 43,  1],
        [58, 52, 43, 57, 57,  2,  1, 57],
        [ 1, 59, 52, 39, 41, 46, 47, 52],
        [43, 42, 50, 39, 56,  6,  1, 50]])

Target Id Batch
tensor([[ 0, 21, 44,  1, 61, 43,  1, 61],
        [52, 43, 57, 57,  2,  1, 57, 43],
        [59, 52, 39, 41, 46, 47, 52, 45],
        [42, 50, 39, 56,  6,  1, 50, 43]])


Next expected token id's index   : tensor(0)
Logit value at the expected index: tensor(0.4160, grad_fn=<SelectBackward0>)

torch.Size([32, 65])
tensor(4.7051, grad_fn=<NllLossBackward0>)

